# Energy Minimization of Lennard-Jones charged particles — Conjugate Gradient

*Utrecht University Molecular Modelling courses from the [Bonvin lab](https://bonvinlab.org).*

This notebook performs a simple **energy minimization (EM)** of a 2D system of
Lennard-Jones particles that may also carry a charge (Coulomb interaction).
Starting from a random arrangement, EM walks the system *downhill* on its potential-energy
surface towards a nearby local minimum. The minimizer here is a **conjugate-gradient**
algorithm (optionally preceded by a few **steepest-descent** steps) with an adaptive step
size.

## Theory in brief

### Lennard-Jones
Written from the squared distance $r^2$, with $Z = (r_{min}^2 / r^2)^3 = (r_{min}/r)^6$:

$$E_{LJ} = \varepsilon\, Z (Z-1)$$

The $Z^2$ term is the steep short-range **repulsion** (overlapping atoms), the $-Z$ term the
weaker long-range **attraction**. The two balance at $r = r_{min}$, where the energy reaches
its minimum $-\varepsilon$: `Epsilon` sets the well *depth* and `Rmin` its *position*.
To save work, pairs beyond a cutoff (`CutOff`) are ignored — this makes the potential
slightly discontinuous at the cutoff, which is harmless here.

### Coulomb

$$E_{Coul} = \frac{q_a q_b}{\epsilon_r\, r}$$

Like charges repel ($E>0$), unlike charges attract ($E<0$); the dielectric constant
`Dielec` ($\epsilon_r$) screens (weakens) the interaction.

### Minimization
The **force** on each atom is minus the gradient of the total energy, i.e. the local
downhill direction.

* **Steepest descent** moves every particle straight along the (normalised) total force
  $\mathbf{F}_k$. Simple, but it tends to *zig-zag* in long narrow valleys.
* **Conjugate gradient** follows a direction that mixes the current force with the previous
  search direction (Fletcher–Reeves), which cancels much of that zig-zag and usually
  converges in far fewer steps:

$$\mathbf{s}_k = \mathbf{F}_k + \gamma\, \mathbf{s}_{k-1}, \qquad
  \gamma = \frac{|\mathbf{F}_k|^2}{|\mathbf{F}_{k-1}|^2}$$

Particles move a step `dr` along the normalised $\mathbf{s}_k$. The first `numsteep` steps
use plain steepest descent (with `numsteep = 0`, the first step self-starts as steepest
descent because $\gamma = 0$). The step is scaled **up** by `alpha` when the energy
decreases and **down** by `beta` when it increases; iteration stops when the energy change,
the step size, or the force norm drops below its threshold.

## 1. Imports

The numerical core uses only the Python **standard library** (`math`, `random`), so it runs
on a bare Python install. **matplotlib** is the one third-party dependency — it draws the
static figures and the trajectory animation (embedded inline as interactive HTML via
`jshtml`). The cell below first **installs matplotlib if it is missing** (handy on Google
Colab), then imports everything; `%matplotlib inline` renders figures inside the notebook.

In [ ]:
# --- Install required packages if missing (e.g. on Google Colab) ---
import importlib.util, subprocess, sys

for pkg in ["matplotlib"]:
    if importlib.util.find_spec(pkg) is None:
        print(f"Installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
    else:
        print(f"{pkg} already available")

from math import sqrt

import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.animation import FuncAnimation
from matplotlib import rc

from random import random, seed

# Show animations inline
rc('animation', html='jshtml')
%matplotlib inline

## 2. Helper functions

Small utilities used throughout:

* `dist` / `dist2` — Euclidean distance and its square (the squared form avoids a needless
  `sqrt` when we only need to compare distances).
* `SignR(a, b)` — returns `a` with the sign of `b`. It implements the **minimum-image
  convention**: the combination `tmp - SignR(halfbox, tmp-halfbox) - SignR(halfbox, tmp+halfbox)`
  wraps a coordinate difference into the range $[-\tfrac{box}{2}, +\tfrac{box}{2}]$, so each
  atom interacts with the *nearest periodic copy* of its neighbours (periodic boundary
  conditions).
* `charge_color` — purely cosmetic: white for positive charges, dark for negative, used when
  drawing the particles.

In [ ]:
### distance ###
def dist(A, B):
    return sqrt((A[0]-B[0])**2 + (A[1]-B[1])**2)

### squared distance ###
def dist2(A, B):
    return (A[0]-B[0])**2 + (A[1]-B[1])**2

### change sign ###
def SignR(a, b):
    if b > 0:
        return a
    else:
        return -a

### colour particles based on charge ###
def charge_color(charge, qat):
    if charge == qat:
        return "#FFFFFF"   # positive
    else:
        return "#333333"   # negative

## 3. Energy functions

Total energy is the sum over all particle pairs of the Lennard-Jones and Coulomb
contributions, using the **nearest image** convention (periodic boundary conditions).

Distances are handled as **squared** distances (`distsquare`) throughout the inner loops:
this avoids computing a `sqrt` for every pair, and lets the cutoff test (`distsquare <
cutoffsquare`) skip distant pairs cheaply. A `sqrt` is taken only where a term actually
needs $r$ itself (the Coulomb $1/r$).

In [ ]:
# LJ energy from the squared distance
def LJ2(distsquare, epsilon, rmin_exp6):
    Z = (1/distsquare)**3 * rmin_exp6
    return epsilon * Z * (Z - 1)

# classical Coulomb from the squared distance
def Coulomb2(r, dielec, qa, qb):
    return qa*qb / (dielec*sqrt(r))

# Calculate energy Evdw + Ecoulomb (uses squared distance), with periodic boundary conditions
def Calc_Ene2(coord, epsilon, rmin, dielec, cutoffsquare, boxdim, elec=1):
    Ene = 0.0
    ELJ = 0.0
    ECoul = 0.0
    rmin_exp6 = rmin**6
    # doubly nested loop over all particle pairs
    for i in range(len(coord)-1):
        for j in range(i+1, len(coord)):
            # squared atomic distance (nearest image)
            distsquare = 0
            for k in range(2):
                tmp = coord[j][k] - coord[i][k]
                halfbox = boxdim[k]/2
                tmp = tmp - SignR(halfbox, tmp-halfbox) - SignR(halfbox, tmp+halfbox)
                distsquare += tmp**2
            if distsquare < cutoffsquare:
                qa = coord[i][2]
                qb = coord[j][2]
                vdw = LJ2(distsquare, epsilon, rmin_exp6)
                Ene += vdw
                ELJ += vdw
                if elec:
                    CC = Coulomb2(distsquare, dielec, qa, qb)
                    Ene += CC
                    ECoul += CC
    return Ene, ELJ, ECoul

## 4. Force functions

The force on an atom is minus the gradient of the total energy — the local *downhill*
direction. For the Lennard-Jones term the component along $x$ is obtained by the chain rule,

$$F_x = -\frac{\partial E}{\partial x} = -\frac{\partial E}{\partial Z}\,
        \frac{\partial Z}{\partial r}\,\frac{\partial r}{\partial x},$$

which maps directly onto the code:

* `dedz` $= \partial E/\partial Z = \varepsilon\,(2Z-1)$
* `dzdr` $= \partial Z/\partial r = -6\,r_{min}^{6}/r^{7}$
* `drdx` $= x_i/r$, where `xi` $= x_j - x_i$ — using the $j-i$ difference already carries the
  sign that turns $-\partial E/\partial x_i$ into the force on atom $i$.

The Coulomb force follows the same pattern, with `dedr` $= -q_a q_b/(\epsilon_r\, r^{2})$.

In [ ]:
# LJ force component (uses squared distance)
def ForceLJ2(distsquare, epsilon, rmin_exp6, xi):
    rij = sqrt(distsquare)
    Z = (1/distsquare)**3 * rmin_exp6
    dedz = epsilon*(2*Z - 1)
    dzdr = rmin_exp6*(-6.0/rij**7.0)
    drdx = xi/rij
    return dedz*dzdr*drdx

# Coulomb force component (uses squared distance)
def ForceCoulomb2(distsquare, dielec, qa, qb, xi):
    rij = sqrt(distsquare)
    dedr = -1.0*(qa*qb/dielec)*(1/distsquare)
    drdx = xi/rij
    return dedr*drdx

# Total force on each atom from Evdw + Ecoulomb (uses squared distance)
def Calc_Force2(coord, epsilon, rmin, dielec, cutoffsquare, boxdim):
    Force = []
    rmin_exp6 = rmin**6
    for i in range(len(coord)):
        tmpforce = [0.0, 0.0]
        for j in range(len(coord)):
            if i == j:
                continue
            # squared atomic distance (nearest image)
            distsquare = 0
            for k in range(2):
                tmp = coord[j][k] - coord[i][k]
                halfbox = boxdim[k]/2
                tmp = tmp - SignR(halfbox, tmp-halfbox) - SignR(halfbox, tmp+halfbox)
                distsquare += tmp**2
            if distsquare < cutoffsquare:
                qa = coord[i][2]
                qb = coord[j][2]
                fflist = []
                for k in range(2):
                    tmp = coord[j][k] - coord[i][k]
                    ff = ForceLJ2(distsquare, epsilon, rmin_exp6, tmp)
                    ff += ForceCoulomb2(distsquare, dielec, qa, qb, tmp)
                    fflist.append(ff)
                for k in range(2):
                    tmpforce[k] = tmpforce[k] + fflist[k]
        Force.append(tmpforce)
    return Force

## 5. The minimizers: steepest descent & conjugate gradient

**Why two?** Steepest descent always steps along the local force, so in a long, narrow
energy valley it overshoots and *zig-zags*, wasting steps. Conjugate gradient corrects the
new direction with a fraction $\gamma$ of the previous one (here the **Fletcher–Reeves**
choice), keeping successive directions roughly *conjugate* and largely eliminating the
zig-zag — so it typically reaches the minimum in many fewer iterations.

Both functions take the current coordinates, the step size `drstep` and the forces, and
return the new coordinates, the force norm, and the **unit search directions** used
(`sdir`):

* `Steepest_descent` moves along the normalised total force.
* `Conjugate_gradient` builds the Fletcher–Reeves conjugate direction from the current
  force and the previous step's force/direction (`forceprev`, `sdirprev`), then normalises
  and moves along it.

Returning `sdir` (instead of using globals as the original GUI did) lets the run loop carry
the state it needs from one step to the next.

In [ ]:
def Steepest_descent(atom_coord, drstep, force):
    """One steepest-descent step.

    Returns
    -------
    new_coord, normf, sdir
        sdir holds the per-atom unit search directions actually used.
    """
    newlist = []
    sdir = []

    # 1) norm of the total force vector
    normf = 0.0
    for i in range(len(atom_coord)):
        normf = normf + force[i][0]**2.0 + force[i][1]**2.0
    normf = sqrt(normf)

    # 2) move every particle along the normalised force
    for i in range(len(atom_coord)):
        q = atom_coord[i][2]
        r0x = atom_coord[i][0]
        r0y = atom_coord[i][1]
        sx = sy = 0.0
        if normf > 0:
            sx = force[i][0]/normf
            sy = force[i][1]/normf
            r0x = r0x + drstep*sx
            r0y = r0y + drstep*sy
        sdir.append([sx, sy])
        newlist.append([r0x, r0y, q])
    return newlist, normf, sdir


def Conjugate_gradient(atom_coord, drstep, force, forceprev, sdirprev):
    """One conjugate-gradient (Fletcher-Reeves) step.

    Parameters
    ----------
    forceprev : forces from the previous step
    sdirprev  : unit search directions from the previous step

    Returns
    -------
    new_coord, normf, sdir
    """
    newlist = []
    sdir = []

    # 1) squared norms of the current and previous total force vectors
    normf2 = 0.0
    normf2prev = 0.0
    for i in range(len(atom_coord)):
        normf2 = normf2 + force[i][0]**2.0 + force[i][1]**2.0
        normf2prev = normf2prev + forceprev[i][0]**2.0 + forceprev[i][1]**2.0

    # Fletcher-Reeves coefficient (gamma = 0 on the first step -> pure steepest descent)
    if normf2prev > 0:
        gamma = normf2/normf2prev
    else:
        gamma = 0.0
    normf = sqrt(normf2)

    # 2) build the conjugate direction, normalise, and move
    for i in range(len(atom_coord)):
        q = atom_coord[i][2]
        r0x = atom_coord[i][0]
        r0y = atom_coord[i][1]

        sx = force[i][0] + gamma*sdirprev[i][0]
        sy = force[i][1] + gamma*sdirprev[i][1]

        normsdir = sqrt(sx**2.0 + sy**2.0)
        if normsdir > 0:
            sx = sx/normsdir
            sy = sy/normsdir
            r0x = r0x + drstep*sx
            r0y = r0y + drstep*sy
        sdir.append([sx, sy])
        newlist.append([r0x, r0y, q])
    return newlist, normf, sdir

## 6. Parameters

These are the same parameters exposed by the sliders/entry boxes of the original GUI.
Change any of them and re-run **this cell together with the Initialisation and Run cells just below** to explore their effect (or use *Kernel → Restart & Run All*). Values are in the
toy model's arbitrary units (lengths in box/canvas units, energies loosely in kcal/mol).

### System and its properties

| Parameter | Meaning | Typical value / range |
|---|---|---|
| `nAtoms` | number of particles | 2–40 |
| `Radius` | particle radius (drawn size, and default absolute charge) | 10–40 — must leave room to place all atoms, or initialisation fails |
| `Rmin` | position of the LJ energy minimum | `2.24 * Radius` |
| `BoxDim` | box dimensions (periodic) | `[500, 500]` |
| `Epsilon` | LJ well depth | 1–100 |
| `Dielec` | dielectric constant (charge screening) | 1 (vacuum) – 80 (water) |
| `qat` | absolute charge per atom | defaults to `Radius` |
| `frac_neg` | fraction of negative charges | 0–1 |
| `CutOff` | non-bonded cutoff distance | 250 |

### Minimizer

| Parameter | Meaning | Typical value / range |
|---|---|---|
| `drinit` | initial step size `dr` | 0.1–5 |
| `drmin` / `drmax` | min / max allowed `dr` | 1e-5 / 5 |
| `alpha` / `beta` | scale `dr` up / down after each step | 1.05 / 0.90 |
| `deltaE` | energy-change stop threshold (tighter -> settles into a deeper minimum) | 1e-5 |
| `normFmin` | force-norm stop threshold | 1e-4 |
| `numsteep` | initial steepest-descent steps before conjugate gradient | 0+ |
| `max_iter` | hard cap on the number of steps | 2000 |

In [ ]:
nAtoms  = 20              # number of atoms
Radius  = 25.0            # atom radius (must be in a sensible range vs nAtoms so all atoms fit)
Rmin    = 2.24 * Radius   # distance at which the LJ energy is minimal
BoxDim  = [500, 500]      # box dimensions
Epsilon = 25.0            # LJ well depth
Dielec  = 1.0             # dielectric constant
qat     = Radius          # atom absolute charge
frac_neg = 0.5            # fraction of negative charges
OverlapFr = 0.0           # fraction of overlap allowed when placing atoms
CutOff  = 250             # non-bonded cutoff
CutOffSquare = CutOff**2

# --- minimizer controls ---
drinit  = 1.00        # initial dr for EM
drmin   = 0.00001     # minimum dr value to keep stepping
drmax   = 5.00        # maximum dr
alpha   = 1.05        # scale factor for dr when Enew < Eold
beta    = 0.90        # scale factor for dr when Enew > Eold
deltaE  = 0.00001     # energy-difference threshold to stop EM (tight -> avoids stopping early on a plateau)
normFmin = 0.0001     # minimum force norm to keep stepping

numsteep = 0          # number of initial steepest-descent steps before conjugate gradient
Seed    = 100         # random number seed
max_iter = 2000       # safety cap on the number of EM steps

## 7. Initialisation

Generate random, non-overlapping starting positions and assign charges
(a fraction `frac_neg` negative, the rest positive).

In [ ]:
import sys

### generate random, non-overlapping coordinates ###
def InitConf(n, dim, radius, qat, frac_neg):
    seed(Seed)
    print("Initializing box, please wait...")
    tmp_coord = []
    i = 0
    ntrial = 0
    nneg = int(float(n) * frac_neg)
    npos = n - nneg

    # first atom
    x = random()*(dim[0]-2*radius) + radius
    y = random()*(dim[1]-2*radius) + radius
    charge = -qat
    if npos == n:
        charge = qat
    i += 1
    if n == 2:
        tmp_coord.append([175, 300, charge])
    else:
        tmp_coord.append([x, y, charge])

    # remaining negative charges
    while i < nneg:
        x = random()*(dim[0]-2*radius) + radius
        y = random()*(dim[1]-2*radius) + radius
        OVERLAP = 1
        for j in range(i):
            if dist(tmp_coord[j], [x, y]) < (1-OverlapFr)*2*radius:
                OVERLAP = 0
        if OVERLAP:
            charge = -qat
            if n == 2:
                tmp_coord.append([325, 300, charge])
            else:
                tmp_coord.append([x, y, charge])
            i += 1
        ntrial += 1
        if ntrial > 100000:
            print("initialisation failed")
            print("==> reduce radius or number of atoms")
            sys.exit()

    # remaining positive charges
    while i < n:
        x = random()*(dim[0]-2*radius) + radius
        y = random()*(dim[1]-2*radius) + radius
        OVERLAP = 1
        for j in range(i):
            if dist(tmp_coord[j], [x, y]) < (1-OverlapFr)*2*radius:
                OVERLAP = 0
        if OVERLAP:
            charge = qat
            if n == 2:
                tmp_coord.append([325, 300, charge])
            else:
                tmp_coord.append([x, y, charge])
            i += 1
        ntrial += 1
        if ntrial > 10**10:
            print("initialisation failed")
            print("==> reduce radius or number of atoms")
            sys.exit()
    return tmp_coord


Atom_Coord = InitConf(nAtoms, BoxDim, Radius, qat, frac_neg)
Color = [charge_color(a[2], qat) for a in Atom_Coord]
print(f"Placed {len(Atom_Coord)} atoms.")

## 8. Run the minimization

At each step the minimizer computes
the forces, takes a minimization step — **steepest descent for the first `numsteep` steps
(and the very first step), conjugate gradient thereafter** — adaptively rescales `dr`,
applies periodic boundary conditions, and records the trajectory and energies. It carries
the previous force and search direction (`forceprev`, `sdirprev`) forward for the conjugate-
gradient update.

It stops as soon as **any** of three convergence criteria is met — the energy change falls
below `deltaE`, the step size drops below `drmin`, or the average force norm falls below
`normFmin` — or after at most `max_iter` steps.

In [ ]:
def run_minimization(Atom_Coord):
    """Headless minimization: `numsteep` steepest-descent steps, then conjugate gradient.
    Returns trajectory + energy history."""
    coord = [list(a) for a in Atom_Coord]   # work on a copy
    drstep = drinit

    # force / search-direction state carried between steps
    forceprev = [[0.0, 0.0] for _ in coord]
    sdirprev  = [[0.0, 0.0] for _ in coord]

    Ene, EneLJ, EneCoul = Calc_Ene2(coord, Epsilon, Rmin, Dielec, CutOffSquare, BoxDim)
    Ene_prev = Ene

    traj    = [ [list(a) for a in coord] ]   # snapshot per step
    E_hist  = [Ene]
    Elj_hist = [EneLJ]
    Ecoul_hist = [EneCoul]

    print("Iteration: %8d Epot: %6.1f Elj: %6.1f Ecoul: %6.1f" % (0, Ene, EneLJ, EneCoul))

    for step in range(1, max_iter+1):
        Force = Calc_Force2(coord, Epsilon, Rmin, Dielec, CutOffSquare, BoxDim)

        # steepest descent for the first `numsteep` steps (and the very first step),
        # conjugate gradient afterwards
        if step <= numsteep or step == 1:
            coord, normF, sdir = Steepest_descent(coord, drstep, Force)
        else:
            coord, normF, sdir = Conjugate_gradient(coord, drstep, Force, forceprev, sdirprev)

        Ene, EneLJ, EneCoul = Calc_Ene2(coord, Epsilon, Rmin, Dielec, CutOffSquare, BoxDim)
        Ene_diff = Ene - Ene_prev

        # adaptive step size
        if Ene_diff < 0.0:
            drstep = min(drmax, drstep*alpha)
        else:
            drstep = drstep*beta
        Ene_prev = Ene

        # remember this step's force and search direction for the next CG step
        forceprev = Force
        sdirprev = sdir

        # periodic boundary conditions
        for pp in range(len(coord)):
            for k in range(2):
                if coord[pp][k] < 0:
                    coord[pp][k] += BoxDim[k]
                if coord[pp][k] > BoxDim[k]:
                    coord[pp][k] -= BoxDim[k]

        normF = normF/len(coord)

        traj.append([list(a) for a in coord])
        E_hist.append(Ene)
        Elj_hist.append(EneLJ)
        Ecoul_hist.append(EneCoul)

        if step % 50 == 0:
            print("Iteration: %8d Epot: %6.1f Elj: %6.1f Ecoul: %6.1f deltaE: %10.6f <normF>: %8.6f dr: %8.6f"
                  % (step, Ene, EneLJ, EneCoul, Ene_diff, normF, drstep))

        # convergence
        if abs(Ene_diff) < deltaE or drstep < drmin or normF < normFmin:
            print("STOPPING... deltaE<%g, or drstep<%g, or normF<%g" % (deltaE, drmin, normFmin))
            print("Iteration: %8d Epot: %6.1f Elj: %6.1f Ecoul: %6.1f deltaE: %10.6f <normF>: %8.6f dr: %8.6f"
                  % (step, Ene, EneLJ, EneCoul, Ene_diff, normF, drstep))
            break

    return traj, E_hist, Elj_hist, Ecoul_hist


traj, E_hist, Elj_hist, Ecoul_hist = run_minimization(Atom_Coord)
print(f"\nDone in {len(traj)-1} steps. Final Epot = {E_hist[-1]:.2f}")

## 9. Energy convergence

How the total, Lennard-Jones and Coulomb energies evolve during the minimization.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
steps = range(len(E_hist))
ax.plot(steps, E_hist,     label="Epot (total)", lw=2)
ax.plot(steps, Elj_hist,   label="E$_{LJ}$",  lw=1.5)
ax.plot(steps, Ecoul_hist, label="E$_{Coul}$", lw=1.5)
ax.set_xlabel("EM step")
ax.set_ylabel("Energy (kcal/mol)")
ax.set_title("Steepest-descent energy minimization")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 10. Visualise the system

Start and end configurations side by side. White = positive charge, dark = negative.

In [ ]:
def draw_config(ax, coord, title):
    ax.set_xlim(0, BoxDim[0])
    ax.set_ylim(0, BoxDim[1])
    ax.set_aspect('equal')
    ax.set_facecolor("#ccddff")
    ax.set_title(title)
    ax.invert_yaxis()   # match the original canvas (y downwards)
    for a in coord:
        col = charge_color(a[2], qat)
        ax.add_patch(Circle((a[0], a[1]), Radius, facecolor=col, edgecolor="black", lw=0.8))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5.5))
draw_config(ax1, traj[0],  f"Initial  (Epot = {E_hist[0]:.1f})")
draw_config(ax2, traj[-1], f"Minimized (Epot = {E_hist[-1]:.1f})")
plt.tight_layout()
plt.show()

## 11. Animation of the minimization

Replays the whole trajectory. This reproduces the live view of the original GUI.
(To keep it light, only every few frames are shown — adjust `stride`.)

In [ ]:
stride = max(1, len(traj)//120)   # cap at ~120 frames
frames = list(range(0, len(traj), stride))

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(0, BoxDim[0])
ax.set_ylim(0, BoxDim[1])
ax.set_aspect('equal')
ax.set_facecolor("#ccddff")
ax.invert_yaxis()

circles = [Circle((a[0], a[1]), Radius,
                  facecolor=charge_color(a[2], qat), edgecolor="black", lw=0.8)
           for a in traj[0]]
for c in circles:
    ax.add_patch(c)
title = ax.set_title("")

def update(frame_idx):
    f = frames[frame_idx]
    for c, a in zip(circles, traj[f]):
        c.center = (a[0], a[1])
    title.set_text(f"step {f}   Epot = {E_hist[f]:.1f}")
    return circles + [title]

anim = FuncAnimation(fig, update, frames=len(frames), interval=80, blit=False)
plt.close(fig)   # avoid a duplicate static figure
anim

## 12. Comparison of the three minimizers

This notebook implements **the conjugate-gradient minimizer (with an optional steepest-descent warm-up)**. The three companion notebooks
(`LJ-ELEC_EM-steepest_Py3`, `LJ-ELEC_EM-conjugate_Py3`, `simplex`) minimize the **same system**
— 20 particles with identical parameters and the same random seed (`Seed = 100`). Running each
**with its default parameters** gives:

| Method | Final energy | Steps |
|---|---|---|
| Steepest descent | ≈ −409 | ~1100 |
| Conjugate gradient | ≈ −383 | ~970 |
| Simplex (Nelder–Mead) | ≈ −187 | ~620 |

Take-aways:

* All three reach a **local** minimum — none is guaranteed to find the global minimum, and the
  result depends on the starting configuration (the random seed) and the path taken.
* **Steepest descent** lands in the deepest basin here; **conjugate gradient** takes fewer steps
  but, following a different path, settles in a slightly shallower minimum.
* The **simplex** is *derivative-free* (energy only, no forces) — simple and robust, but it scales
  poorly to this 40-dimensional search space (2 × 20 coordinates), so it converges to a much
  shallower minimum. A good illustration of why gradient-based methods dominate for smooth,
  high-dimensional problems.

*(Energies and step counts are for `Seed = 100` with each notebook's default parameters; other
seeds give different absolute numbers.)*